# HarvestGuard Rwanda — Model V3 candidate, Colab

**For aggregation with your existing trained model, use [HarvestGuard_Quantity_Aggregation_Colab.ipynb](HarvestGuard_Quantity_Aggregation_Colab.ipynb). This development notebook retrains and is not needed for an aggregation-only change.**

**Purpose:** predict each profile's expected loss percentage, convert it to kg using that profile's production, and rank district/crop/season groups by total estimated kg lost. The rate model still targets percentages.

1. Open this notebook in Google Colab and connect your Drive.
2. Check the configuration cell and use **Runtime → Run all**.
3. Read the 2025 comparison before deciding whether the candidate is better.
4. Download the candidate ZIP. The dashboard supports the new quantity columns; replace its saved V3 bundle only after reviewing the run.

The notebook contains its own modeling module: no companion upload or API key is needed.
The real CSV stays in your Drive. The author could review V2 code/exports locally but did not have that CSV;
new real-data accuracy is therefore **unknown until your Colab run finishes**.

2024 supplies eligibility, fitting, calibration, tuning and selection. 2025 is evaluated after the choice is locked.
Because 2025 has been inspected in earlier project versions, call it **temporal validation**, not an untouched blind test.
A new uninspected period is still needed for a convincing final generalization claim.

## What the V2 review found

The supplied legacy V2 exports report these **2025 A+B loss-rate ranking** results. These are not kg-based priority metrics:

| Model | Profile MAE (percentage points) | Macro district NDCG@5 | Top-five overlap |
|---|---:|---:|---:|
| Hurdle XGBoost V2 | 0.4976 | 0.1996 | 0.2267 |
| Exact 2024 historical baseline | 0.5003 | 0.2580 | 0.2667 |

The model slightly improves numerical error but loses on district ranking. Its 2025 R² is negative.
V2's highest predicted decile averages 1.1522% predicted versus 0.2484% observed;
these saved results point to poor temporal discrimination/calibration, not just a need for more trees.
About 92.92% of eligible 2024 profiles have zero reported losses.

Changes tested here:

- Strong exact and smoothed historical baselines are included in model selection; a baseline may win.
- Early stopping and calibration use subsets **inside** the training fold, never the outer validation labels.
- Occurrence and positive severity have separate depth/min-child settings; severity can use Gamma or Tweedie loss.
- TPE runs beyond its explicitly smaller random-startup budget. V2's 4–5 trials were below its default 10 startup trials.
- Baseline/ML blends and calibration are tested using 2024 only.
- Rankings use supported groups and tie-aware evaluation. Constant observed cohorts are marked uninformative.

These are candidate improvements, not a promise of higher scores. Metrics in this notebook are generated by your run;
there is no hard-coded V3 success claim.

In [ ]:
# @title 1. Environment and Google Drive
import sys, subprocess, importlib.util
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'numpy>=1.26,<3', 'pandas>=2.2,<4', 'scipy>=1.11,<2',
        'scikit-learn>=1.5,<2', 'xgboost>=2.1,<4', 'optuna>=4,<5', 'joblib', 'matplotlib'])
    from google.colab import drive
    drive.mount('/content/drive')
print('Colab:', IN_COLAB)

## Configuration — check before running

The default **at_harvest** feature policy excludes realized weather after harvest.
It is appropriate only if the other supplied features, including planned harvest month, are known at the intended prediction time.
Past-weather joins still need checking against their actual issue/observation dates.
If your decision is retrospective, choose `retrospective`; that permits the supplied realized post-harvest weather.
Do not describe that experiment as a live forecast. Genuine archived forecasts are needed to assess a forecast-based deployment.

The default run uses 12 trials and up to 500 trees, with early stopping. This is a first-pass CPU budget,
not an exhaustive search. Increase to 30–40 trials only after checking data quality and the initial run.
Trials checkpoint after completion, so rerunning the same configuration can resume.
Do not change settings because of 2025 scores and still call 2025 an untouched test.

`PRODUCTION_COLUMN` must identify the original production denominator for each profile/crop/season. Restore it to the CSV if cleaning removed it; never infer production from area or profile count. Set `PRODUCTION_UNIT` to `kg` or metric `tonnes`. Production must be complete, positive and match the loss target's scope. It is used after rate prediction, not added as a model feature.

`SURVEY_WEIGHT_COLUMN` is optional: use only verified survey expansion weights. Without weights, kg totals cover the sampled profiles and do not estimate whole districts or national losses. Model fitting weights are not survey weights. For operational forecasts, production must be known or explicitly forecast at the decision date; realized historical production supports historical validation only.

`ENTITY_COLUMN`: set a verified household/farmer ID if available; it must be stable and unique within district.
Otherwise the original district × crop × season grouping is retained. These splits answer different generalization questions;
neither substitutes for another held-out year.

In [ ]:
# @title 2. Run configuration
DATA_PATH = Path('/content/drive/MyDrive/NISR/postharvest_regression_model_ready_2024_2025_with_production.csv')
if not IN_COLAB:
    DATA_PATH = Path('postharvest_regression_model_ready_2024_2025_with_production.csv')

OUTPUT_DIR = Path('/content/drive/MyDrive/NISR/HarvestGuard/model_v3_retrained_quantity') if IN_COLAB else Path('model_v3_retrained_quantity')
WORK_DIR = Path('/content/harvestguard_v3_work') if IN_COLAB else Path('harvestguard_v3_work')
FEATURE_MODE = 'at_harvest'  # 'retrospective' is a separate, explicitly historical experiment.
TRIALS = 12
MAX_TREES = 500
N_SPLITS = 3
MIN_PROFILES_PER_DISTRICT = 10
THREADS = 4
ENTITY_COLUMN = None  # Example only: 'household_id' if present, verified and complete.
PRODUCTION_COLUMN = 'total_crop_production'  # Verified in the supplied production-bearing CSV; kg confirmed by the user.
PRODUCTION_UNIT = 'kg'  # 'kg' or metric 'tonnes'; converted once to kg.
SURVEY_WEIGHT_COLUMN = None  # Verified expansion weights only; None gives sample totals.
SOURCE_QUANTITIES = None  # Optional: {'production': '...', 'storage': '...', 'processing': '...', 'packaging': '...'}
AUTO_DOWNLOAD = True

if OUTPUT_DIR.name == 'model_v2':
    raise ValueError('Use a new candidate directory. Do not overwrite V2 before reviewing the results.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('Feature policy:', FEATURE_MODE)
print('Data:', DATA_PATH)
print('Candidate outputs:', OUTPUT_DIR)

In [ ]:
# @title 3. Load the included candidate pipeline (expand to inspect)
MODULE_SOURCE = '"""Colab candidate pipeline: expected loss rates, honest baselines and ranking.\n\nThis is a research candidate, separate from the running Streamlit application.\nThe model predicts percentages (0 to 100). Intervention ranking sums profile\nloss quantities after multiplying each prediction by its production denominator.\n"""\n\nfrom pathlib import Path\nimport hashlib\nimport json\nimport os\nimport platform\nimport time\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport scipy\nfrom scipy.special import expit, logit\nfrom scipy.stats import spearmanr\nimport sklearn\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, brier_score_loss, mean_tweedie_deviance, ndcg_score\nfrom sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import OneHotEncoder\nimport xgboost\nfrom xgboost import XGBClassifier, XGBRegressor\n\n\nTARGET = "actionable_loss_rate_pct"\nKEYS = ["season", "crop_name", "district"]\nCATS = ["season", "district", "crop_name", "farmer_type", "harvest_month"]\nBEFORE = ["n_plots_crop", "crop_plot_area_ha_total", "num_crops_in_plot", "rain_30d_before_mm", "humidity_mean_7d_before_pct"]\nAFTER = ["rain_14d_after_mm", "rainy_days_14d_after", "humidity_mean_14d_after_pct", "temperature_mean_14d_after_c"]\nSEED = 42\nCODE_VERSION = "harvestguard-v3-candidate-2"\n\n\ndef versions():\n    return {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,\n            "scipy": scipy.__version__, "sklearn": sklearn.__version__, "xgboost": xgboost.__version__}\n\n\ndef load_saved_rate_model(path):\n    """Load a trusted saved bundle for inference, including Linux-to-Windows IO.\n\n    XGBoost issue #12459 documents a nonportable RNG state in pickled boosters.\n    For that exact Windows error only, recover the unchanged Model section via\n    XGBoost\'s model loader. Training configuration/RNG state is not required for\n    inference. No trees, objectives, calibration or preprocessing are refitted.\n    https://github.com/dmlc/xgboost/issues/12459\n    """\n    try:\n        fitted = joblib.load(path)\n        fitted._quantity_loading_method = "original_joblib"\n        return fitted\n    except xgboost.core.XGBoostError as error:\n        if os.name != "nt" or "input stream corrupted" not in str(error):\n            raise\n    original = xgboost.Booster.__setstate__\n    recovered = []\n\n    def restore_model_only(booster, state):\n        snapshot = bytes(state["handle"])\n        # XGBoost\'s UBJSON snapshot has Config then Model at the top level.\n        # Pass the original Model bytes directly to the native model loader:\n        # no tree values are decoded, edited or reserialized by Python.\n        marker = b"L" + (5).to_bytes(8, "big", signed=True) + b"Model"\n        learner = b"{L" + (7).to_bytes(8, "big", signed=True) + b"learner"\n        if snapshot[:1] != b"{" or snapshot[-1:] != b"}" or snapshot.count(marker) != 1:\n            raise ValueError("The saved XGBoost snapshot is not the supported Config/Model UBJSON layout. Use the original Colab environment to export portable model files.")\n        start = snapshot.index(marker) + len(marker)\n        model_buffer = snapshot[start:-1]  # Remove only the snapshot\'s outer closing brace.\n        if not model_buffer.startswith(learner) or model_buffer[-1:] != b"}":\n            raise ValueError("The saved snapshot has no complete final Model section. Use the original Colab environment to export portable model files.")\n        xgboost.Booster.__init__(booster, model_file=bytearray(model_buffer))\n        booster.__dict__.update({key: value for key, value in state.items() if key != "handle"})\n        recovered.append(1)\n\n    # The replacement is scoped to loading this bundle and is always restored.\n    try:\n        xgboost.Booster.__setstate__ = restore_model_only\n        fitted = joblib.load(path)\n    finally:\n        xgboost.Booster.__setstate__ = original\n    if not recovered:\n        raise ValueError("No saved boosters were recovered; the original loading error remains unresolved.")\n    fitted._quantity_loading_method = "existing_model_sections_from_windows_incompatible_rng_snapshot"\n    fitted._quantity_inference_only = True\n    return fitted\n\n\ndef feature_columns(mode="at_harvest"):\n    if mode not in {"at_harvest", "retrospective"}:\n        raise ValueError("Choose at_harvest or retrospective; never silently mix feature policies.")\n    return CATS + BEFORE + (AFTER if mode == "retrospective" else [])\n\n\ndef prepare_features(raw, features):\n    """The same covariate normalization for training and unlabeled inference."""\n    missing = set(features).difference(raw.columns)\n    if missing:\n        raise ValueError("Required columns missing: " + ", ".join(sorted(missing)))\n    frame = raw.copy()\n    for name in CATS:\n        if name == "harvest_month":\n            month = pd.to_numeric(frame[name], errors="coerce")\n            month = month.where(month.isin(range(1, 13)))\n            frame[name] = month.astype("Int64").astype("string").fillna("_MISSING_").astype(str)\n        else:\n            frame[name] = frame[name].astype("string").str.strip().fillna("_MISSING_").astype(str)\n    for name in set(features).difference(CATS):\n        frame[name] = pd.to_numeric(frame[name], errors="coerce").replace([np.inf, -np.inf], np.nan)\n    if frame[KEYS].eq("_MISSING_").any().any() or frame[KEYS].eq("").any().any():\n        raise ValueError("District, crop and season identities cannot be missing.")\n    return frame.reset_index(drop=True)\n\n\ndef prepare_data(raw, mode="at_harvest"):\n    """Validate rates without imputing missing outcomes or discarding true zeros."""\n    if TARGET not in raw or "year" not in raw:\n        raise ValueError("Development data requires observed targets and year.")\n    frame = prepare_features(raw, feature_columns(mode))\n    frame[TARGET] = pd.to_numeric(frame[TARGET], errors="coerce")\n    if not np.isfinite(frame[TARGET].to_numpy()).all() or not frame[TARGET].between(0, 100).all():\n        raise ValueError("Targets must be observed finite percentages in [0,100]. Audit missing outcomes and units; do not fill them with zero or clip labels.")\n    frame["year"] = pd.to_numeric(frame["year"], errors="raise").astype(int)\n    if not {2024, 2025}.issubset(set(frame.year)):\n        raise ValueError("Both 2024 development and 2025 temporal-validation rows are required.")\n    return frame\n\n\ndef audit_target_quantities(raw, columns, tolerance_pp=1e-5):\n    """Optional check only when original quantities share a verified unit/scope."""\n    required = {"production", "storage", "processing", "packaging"}\n    if set(columns) != required:\n        raise ValueError("Map production, storage, processing and packaging to their source column names.")\n    quantities = raw[list(columns.values())].apply(pd.to_numeric, errors="coerce")\n    if not np.isfinite(quantities.to_numpy()).all() or (quantities < 0).any().any():\n        raise ValueError("Source quantities contain missing, non-finite or negative values; resolve their meaning first.")\n    production = quantities[columns["production"]]\n    loss = sum(quantities[columns[k]] for k in ["storage", "processing", "packaging"])\n    if (production <= 0).any():\n        raise ValueError("Zero-production records require an explicit target policy; do not silently divide by zero.")\n    computed = 100 * loss / production\n    existing = pd.to_numeric(raw[TARGET], errors="coerce")\n    bad = ~np.isclose(computed, existing, atol=tolerance_pp, rtol=1e-6)\n    return pd.DataFrame({"recomputed_loss_pct": computed, "saved_loss_pct": existing, "mismatch": bad})\n\n\ndef prepare_production(raw, production_column="production_kg", unit="kg", survey_weight_column=None):\n    """Map verified profile/crop/season production to kg; never infer it from area.\n\n    Production must be the same denominator as TARGET, once per profile. Survey\n    weights, when supplied, must be verified expansion weights, not model weights.\n    Neither production nor the weights are added to the predictive feature list.\n    """\n    factors = {"kg": 1., "tonnes": 1000.}\n    if unit not in factors:\n        raise ValueError("Production unit must be \'kg\' or \'tonnes\' (metric tonnes).")\n    if not production_column or production_column not in raw:\n        available = ", ".join(repr(str(name)) for name in raw.columns)\n        raise ValueError(\n            f"Configured production column {production_column!r} was not found. "\n            f"Available CSV columns: {available}. "\n            "Quantity ranking requires a verified production column for every profile. "\n            "Set PRODUCTION_COLUMN to its exact CSV name and PRODUCTION_UNIT to \'kg\' or \'tonnes\'. "\n            "If production was removed during cleaning, restore the target\'s original production denominator "\n            "using verified profile/crop/season/year identifiers. Renaming an unrelated column, "\n            "using area or profile count, or changing SOURCE_QUANTITIES will not recover production."\n        )\n    frame = raw.copy()\n    production = pd.to_numeric(frame[production_column], errors="coerce").to_numpy(float) * factors[unit]\n    if not np.isfinite(production).all() or (production <= 0).any():\n        raise ValueError("Production must be finite, positive and observed on every profile; do not impute missing production or silently drop rows.")\n    frame["production_kg"] = production\n    if survey_weight_column is not None:\n        if survey_weight_column not in raw:\n            raise ValueError("The configured survey expansion weight column is missing.")\n        weights = pd.to_numeric(raw[survey_weight_column], errors="coerce").to_numpy(float)\n        if not np.isfinite(weights).all() or (weights <= 0).any():\n            raise ValueError("Survey expansion weights must be observed, finite and positive on every profile.")\n        frame["survey_weight"] = weights\n    elif "survey_weight" in frame:\n        raise ValueError("Explicitly configure survey_weight_column to use existing survey_weight values.")\n    return frame\n\n\ndef profile_loss_quantities(frame, predicted):\n    """Convert each rate before aggregation; also works without observed labels."""\n    if frame.empty:\n        raise ValueError("Quantity ranking requires at least one profile.")\n    if "year" in frame and (frame.year.isna().any() or frame.year.nunique() != 1):\n        raise ValueError("Aggregate one year at a time; do not combine the same season across different years.")\n    predicted = np.asarray(predicted, float)\n    if predicted.shape != (len(frame),) or not np.isfinite(predicted).all() or ((predicted < 0) | (predicted > 100)).any():\n        raise ValueError("Predictions must match rows and be finite percentages in [0,100].")\n    if "production_kg" not in frame:\n        raise ValueError("Quantity ranking requires production_kg on every profile.")\n    production = pd.to_numeric(frame.production_kg, errors="coerce").to_numpy(float)\n    weights = pd.to_numeric(frame.survey_weight, errors="coerce").to_numpy(float) if "survey_weight" in frame else np.ones(len(frame))\n    if not np.isfinite(production).all() or (production <= 0).any():\n        raise ValueError("Production must be finite, positive and observed on every profile.")\n    if not np.isfinite(weights).all() or (weights <= 0).any():\n        raise ValueError("Survey expansion weights must be observed, finite and positive.")\n    if frame[KEYS].isna().any().any() or frame[KEYS].astype(str).apply(lambda s: s.str.strip().eq("")).any().any():\n        raise ValueError("District, crop and season identities cannot be missing.")\n    z = frame[KEYS].copy()\n    z["production_kg"] = production\n    z["survey_weight"] = weights\n    z["expected_loss_pct"] = predicted\n    z["predicted_loss_kg"] = (predicted / 100) * production\n    z["weighted_production_kg"] = production * weights\n    z["weighted_predicted_loss_kg"] = z.predicted_loss_kg * weights\n    if TARGET in frame:\n        observed = pd.to_numeric(frame[TARGET], errors="coerce").to_numpy(float)\n        if not np.isfinite(observed).all() or ((observed < 0) | (observed > 100)).any():\n            raise ValueError("Observed loss rates must be finite percentages in [0,100].")\n        z["observed_loss_pct"] = observed\n        z["observed_loss_kg"] = (observed / 100) * production\n        z["weighted_observed_loss_kg"] = z.observed_loss_kg * weights\n    numeric = z.select_dtypes(include="number").to_numpy()\n    if not np.isfinite(numeric).all():\n        raise ValueError("Loss quantity arithmetic overflowed; verify production units and weights.")\n    return z\n\n\ndef aggregate_losses(frame, predicted):\n    """Sum profile losses by season/crop/district, using expansion weights if given.\n\n    Without survey weights the quantities cover the sample only. A group mean\n    percentage times a district production total is not used here.\n    """\n    z = profile_loss_quantities(frame, predicted)\n    fields = dict(production_kg=("weighted_production_kg", "sum"),\n                  predicted_loss_kg=("weighted_predicted_loss_kg", "sum"),\n                  n_profiles=("expected_loss_pct", "size"),\n                  mean_profile_predicted_loss_pct=("expected_loss_pct", "mean"))\n    if TARGET in frame:\n        fields.update(observed_loss_kg=("weighted_observed_loss_kg", "sum"),\n                      mean_profile_observed_loss_pct=("observed_loss_pct", "mean"))\n    result = z.groupby(KEYS, as_index=False).agg(**fields)\n    for kind in ["predicted"] + (["observed"] if TARGET in frame else []):\n        result[f"{kind}_loss_pct"] = 100 * result[f"{kind}_loss_kg"] / result.production_kg\n        result[f"{kind}_loss_tonnes"] = result[f"{kind}_loss_kg"] / 1000\n    if not np.isfinite(result.select_dtypes(include="number").to_numpy()).all():\n        raise ValueError("Aggregated loss quantities must be finite; verify production units and weights.")\n    result["loss_quantity_scope"] = "survey_expanded" if "survey_weight" in frame else "sample_total"\n    result["priority_rank"] = result.groupby(["season", "crop_name"]).predicted_loss_kg.rank(method="min", ascending=False).astype(int)\n    return result\n\n\ndef quantity_results_from_saved_model(raw, model, metadata, production_column=None, unit="kg",\n                                      survey_weight_column=None, year=2025):\n    """Predict and aggregate with an existing model, without fitting or tuning.\n\n    Use the saved eligibility and features. Loss labels are optional and are\n    never passed to the model. Production is required only on the scored year\'s\n    eligible profiles, not on historical training rows that are not rescored.\n    """\n    if metadata.get("target") != TARGET:\n        raise ValueError("The saved model must predict actionable_loss_rate_pct.")\n    features, crops = metadata.get("features"), metadata.get("eligible_crops")\n    if not isinstance(features, list) or not features or not isinstance(crops, list) or not crops:\n        raise ValueError("Saved metadata must contain the original features and eligible_crops.")\n    if set(features) != set(feature_columns(metadata.get("feature_policy", "at_harvest"))):\n        raise ValueError("Saved feature definitions differ from the supported V3 feature policy.")\n    if "year" not in raw or "crop_name" not in raw:\n        raise ValueError("Farmer-level input must include year and crop_name; do not use the district-average prediction CSV.")\n    raw = raw.reset_index(drop=True)\n    years = pd.to_numeric(raw.year, errors="coerce")\n    if not np.isfinite(years.to_numpy(float)).all() or (years % 1 != 0).any():\n        raise ValueError("Input year must be observed whole numbers on every row.")\n    year_rows = raw.loc[years.eq(year)].copy()\n    if year_rows.empty:\n        raise ValueError(f"No farmer-level rows were found for {year}.")\n    crop_names = year_rows.crop_name.astype("string").str.strip()\n    selected = year_rows.loc[crop_names.isin(crops)].copy()\n    selected["crop_name"] = crop_names.loc[selected.index]\n    selected["year"] = year\n    if selected.empty:\n        raise ValueError("No input profiles match the saved model\'s eligible crops.")\n    selected = prepare_production(selected, production_column, unit, survey_weight_column)\n    incoming = prepare_features(selected, features)\n    # Explicit feature selection prevents labels or production from becoming model inputs.\n    rate = np.asarray(model.predict(incoming[features]), float)\n    profiles = profile_loss_quantities(incoming, rate)\n    profiles.insert(0, "year", year)\n    risk = aggregate_losses(incoming, rate)\n    risk["vulnerability_rank"] = risk.priority_rank\n    risk["support_level"] = pd.cut(risk.n_profiles, [-1, 9, 29, 99, np.inf], labels=["Limited", "Low", "Moderate", "Strong"]).astype(str)\n    risk = risk.sort_values(["season", "crop_name", "priority_rank", "district"]).reset_index(drop=True)\n    manifest = {\n        "operation": "existing_model_inference_and_quantity_aggregation",\n        "model_retrained": False,\n        "model_loading_method": getattr(model, "_quantity_loading_method", "provided_existing_model"),\n        "source_candidate_version": metadata.get("candidate_version", "unspecified"),\n        "champion": metadata.get("champion", "unspecified"),\n        "target": TARGET, "features": features, "feature_policy": metadata.get("feature_policy"),\n        "year": int(year), "profiles_in_year": len(year_rows), "profiles_scored": len(selected),\n        "profiles_excluded_unsupported_crop": len(year_rows) - len(selected),\n        "production_source_column": production_column, "production_source_unit": unit,\n        "survey_weight_source_column": survey_weight_column,\n        "ranking_basis": "predicted_loss_kg",\n        "loss_quantity_scope": "survey_expanded" if survey_weight_column is not None else "sample_total",\n        "observed_losses_available": TARGET in selected,\n        "quantity_ranking_validation_recomputed": False,\n        "model_ready_for_national_allocation": False,\n        "note": "Saved model and historical rate-validation reports are unchanged. These results do not establish new ranking accuracy or intervention benefits.",\n    }\n    return profiles, risk, manifest\n\n\ndef crop_evidence(dev):\n    if not dev.year.eq(2024).all():\n        raise ValueError("Eligibility must be defined from 2024 only.")\n    table = dev.groupby("crop_name").agg(n=(TARGET, "size"), positives=(TARGET, lambda y: int((y > 0).sum())),\n                                        districts=("district", "nunique"), seasons=("season", "nunique")).reset_index()\n    table["eligible"] = (table.n >= 200) & (table.positives >= 10) & (table.districts >= 5) & (table.seasons >= 2)\n    table["evidence_warning"] = np.where(table.positives < 30, "Very few positive cases: avoid crop-specific fitting", "")\n    return table\n\n\ndef groups(frame, entity_column=None):\n    if entity_column:\n        if entity_column not in frame or frame[entity_column].isna().any():\n            raise ValueError("The configured entity ID must exist and be observed on every row.")\n        # IDs are assumed unique within district; they are never predictive features.\n        return frame["district"].astype(str) + "|entity|" + frame[entity_column].astype(str)\n    return frame[KEYS].astype(str).agg("|".join, axis=1)\n\n\ndef cv_splits(dev, n_splits=3, entity_column=None):\n    if not dev.year.eq(2024).all():\n        raise ValueError("2025 outcomes must not enter development splits.")\n    group = groups(dev, entity_column)\n    if group.nunique() < n_splits:\n        raise ValueError("Too few independent groups for the requested folds.")\n    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)\n    splits = list(splitter.split(dev, dev[TARGET].gt(0), group))\n    coverage = np.zeros(len(dev), dtype=int)\n    for train, valid in splits:\n        assert set(group.iloc[train]).isdisjoint(group.iloc[valid])\n        coverage[valid] += 1\n    assert (coverage == 1).all()\n    return splits\n\n\ndef aggregate_rates(frame, predicted):\n    if "production_kg" in frame:\n        return aggregate_losses(frame, predicted).rename(columns={"predicted_loss_pct": "predicted", "observed_loss_pct": "observed", "n_profiles": "profiles"})\n    if "survey_weight" in frame:\n        raise ValueError("Survey weights cannot substitute for missing production_kg.")\n    predicted = np.asarray(predicted, float)\n    if len(predicted) != len(frame) or not np.isfinite(predicted).all() or ((predicted < 0) | (predicted > 100)).any():\n        raise ValueError("Predictions must match rows and be finite percentages in [0,100].")\n    z = frame[KEYS + [TARGET]].copy()\n    z["predicted"] = predicted\n    return z.groupby(KEYS, as_index=False).agg(observed=(TARGET, "mean"), predicted=("predicted", "mean"), profiles=(TARGET, "size"))\n\n\ndef _top_membership(values, k):\n    values = np.asarray(values, float)\n    cutoff = np.sort(values)[-k]\n    above, equal = values > cutoff, values == cutoff\n    return above.astype(float) + equal * ((k - above.sum()) / equal.sum())\n\n\ndef ranking_table(agg, min_profiles=10, min_districts=5):\n    quantity = "predicted_loss_kg" in agg\n    observed, predicted = ("observed_loss_kg", "predicted_loss_kg") if quantity else ("observed", "predicted")\n    rows = []\n    for (season, crop), all_rows in agg.groupby(["season", "crop_name"]):\n        part = all_rows.loc[all_rows.profiles >= min_profiles]\n        reason = ""\n        if len(part) < min_districts:\n            reason = "too_few_supported_districts"\n        elif np.ptp(part[observed].to_numpy()) < 1e-12:\n            reason = "constant_observed_losses" if quantity else "constant_observed_rates"\n        row = {"season": season, "crop_name": crop, "districts": len(part), "excluded_reason": reason,\n               "ranking_basis": "predicted_loss_kg" if quantity else "predicted_loss_pct"}\n        if not reason:\n            y, p = part[observed].to_numpy(), part[predicted].to_numpy()\n            k = min(5, len(part))\n            row.update(ndcg5=float(ndcg_score(y[None, :], p[None, :], k=k, ignore_ties=False)),\n                       spearman=0.0 if np.ptp(p) < 1e-12 else float(spearmanr(y, p).statistic),\n                       top5_overlap=float(np.dot(_top_membership(y, k), _top_membership(p, k)) / k))\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef metrics(frame, predicted, min_profiles=10):\n    y, p = frame[TARGET].to_numpy(float), np.asarray(predicted, float)\n    agg = aggregate_rates(frame, p)\n    supported = agg.loc[agg.profiles >= min_profiles]\n    ranks = ranking_table(agg, min_profiles)\n    usable = ranks.loc[ranks.excluded_reason.eq("")]\n    mean_y = float(y.mean())\n    denominator = float(np.sum((y - mean_y) ** 2))\n    result = {\n        "mae_pp": float(np.mean(np.abs(y - p))), "rmse_pp": float(np.sqrt(np.mean((y - p) ** 2))),\n        "r2": float(1 - np.sum((y - p) ** 2) / denominator) if denominator > 0 else np.nan,\n        "mean_predicted_pct": float(p.mean()), "mean_observed_pct": mean_y,\n        "district_mae_pp": float(np.mean(np.abs(supported.observed - supported.predicted))) if len(supported) else np.nan,\n        "district_rmse_pp": float(np.sqrt(np.mean((supported.observed - supported.predicted) ** 2))) if len(supported) else np.nan,\n        "tweedie_deviance_floor_0_001pp": float(mean_tweedie_deviance(y / 100, np.maximum(p, .001) / 100, power=1.5)),\n        "ndcg5_macro": float(usable.ndcg5.mean()) if len(usable) else np.nan,\n        "spearman_macro": float(usable.spearman.mean()) if len(usable) else np.nan,\n        "top5_overlap_macro": float(usable.top5_overlap.mean()) if len(usable) else np.nan,\n        "ranking_cohorts": int(len(usable)), "supported_district_groups": int(len(supported)),\n    }\n    if "predicted_loss_kg" in agg:\n        error = supported.observed_loss_kg - supported.predicted_loss_kg\n        result.update(district_mae_kg=float(np.abs(error).mean()) if len(error) else np.nan,\n                      district_rmse_kg=float(np.sqrt((error ** 2).mean())) if len(error) else np.nan,\n                      total_predicted_loss_kg=float(agg.predicted_loss_kg.sum()),\n                      total_observed_loss_kg=float(agg.observed_loss_kg.sum()))\n    return result\n\n\nclass HistoricalRate:\n    """Exact prior-period means or empirical shrinkage toward broader groups."""\n    LEVELS = [("crop_name",), ("crop_name", "season"), ("district", "crop_name"), ("district", "crop_name", "season")]\n\n    def __init__(self, alpha=0):\n        self.alpha = float(alpha)\n\n    def fit(self, train):\n        y = train[TARGET].to_numpy(float)\n        self.mean_ = float(y.mean())\n        self.prob_ = float((y > 0).mean())\n        self.tables_ = {}\n        z = train.assign(_positive=(y > 0).astype(int))\n        for keys in self.LEVELS:\n            self.tables_[keys] = z.groupby(list(keys), as_index=False).agg(_sum=(TARGET, "sum"), _n=(TARGET, "size"), _pos=("_positive", "sum"))\n        return self\n\n    def _lookup(self, frame, keys, parent_rate, parent_prob):\n        joined = frame[list(keys)].merge(self.tables_[keys], on=list(keys), how="left", sort=False)\n        n = joined._n.fillna(0).to_numpy()\n        denominator = n + self.alpha\n        rate = np.divide(joined._sum.fillna(0).to_numpy() + self.alpha * parent_rate, denominator,\n                         out=np.asarray(parent_rate, float).copy(), where=denominator > 0)\n        prob = np.divide(joined._pos.fillna(0).to_numpy() + self.alpha * parent_prob, denominator,\n                         out=np.asarray(parent_prob, float).copy(), where=denominator > 0)\n        return rate, prob, n > 0\n\n    def predict_components(self, frame):\n        count = len(frame)\n        base, pbase = np.full(count, self.mean_), np.full(count, self.prob_)\n        crop, pcrop, _ = self._lookup(frame, self.LEVELS[0], base, pbase)\n        cs, pcs, cs_known = self._lookup(frame, self.LEVELS[1], crop, pcrop)\n        dc, pdc, dc_known = self._lookup(frame, self.LEVELS[2], crop, pcrop)\n        if self.alpha:\n            parent, pparent = .5 * (cs + dc), .5 * (pcs + pdc)\n        else:\n            parent, pparent = np.where(dc_known, dc, cs), np.where(dc_known, pdc, pcs)\n        rate, prob, _ = self._lookup(frame, self.LEVELS[3], parent, pparent)\n        return np.clip(rate, 0, 100), np.clip(prob, 0, 1), np.divide(rate, prob, out=np.zeros(count), where=prob > 0)\n\n    def predict(self, frame):\n        return self.predict_components(frame)[0]\n\n\ndef preprocessor(features):\n    nums = [c for c in features if c not in CATS]\n    return ColumnTransformer([\n        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=5), CATS),\n        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)), nums),\n    ])\n\n\ndef _holdout(frame, fraction, seed, entity_column=None):\n    group = groups(frame, entity_column)\n    if group.nunique() < 3:\n        raise ValueError("Too few groups for independent early-stopping and calibration subsets.")\n    for attempt in range(20):\n        left, right = next(GroupShuffleSplit(n_splits=1, test_size=fraction, random_state=seed + attempt).split(frame, groups=group))\n        if frame.iloc[left][TARGET].gt(0).sum() >= 8 and frame.iloc[right][TARGET].gt(0).sum() >= 2:\n            assert set(group.iloc[left]).isdisjoint(group.iloc[right])\n            return left, right\n    raise ValueError("Too few positive losses in independent training groups. More data or a simpler baseline is needed.")\n\n\ndef prepare_partition(train, valid, features, seed=SEED, entity_column=None):\n    train = train.reset_index(drop=True)\n    base_idx, cal_idx = _holdout(train, .15, seed, entity_column)\n    base, cal = train.iloc[base_idx].reset_index(drop=True), train.iloc[cal_idx].reset_index(drop=True)\n    core_idx, stop_idx = _holdout(base, .15, seed + 100, entity_column)\n    core, stop = base.iloc[core_idx], base.iloc[stop_idx]\n    core_pp, base_pp = preprocessor(features), preprocessor(features)\n    core_x = core_pp.fit_transform(core[features])\n    base_x = base_pp.fit_transform(base[features])\n    return {"base": base, "cal": cal, "core": core, "stop": stop, "valid": valid,\n            "pp": base_pp, "core_x": core_x, "stop_x": core_pp.transform(stop[features]),\n            "base_x": base_x, "cal_x": base_pp.transform(cal[features])}\n\n\ndef group_weights(frame, mode):\n    if mode == "profiles":\n        return np.ones(len(frame))\n    size = frame.groupby(KEYS)[TARGET].transform("size").to_numpy(float)\n    w = 1 / size\n    return np.clip(w / w.mean(), .25, 4.0)\n\n\nclass ConstantEstimator:\n    def __init__(self, value):\n        self.value = float(value)\n\n    def predict(self, x):\n        return np.full(x.shape[0], self.value)\n\n    def predict_proba(self, x):\n        p = self.predict(x)\n        return np.column_stack([1 - p, p])\n\n\ndef _fit_booster(part, params, stage, max_trees=500, threads=4):\n    classification = stage == "occurrence"\n    positive_only = stage == "severity"\n    core_y, stop_y, base_y = [part[k][TARGET].to_numpy(float) for k in ["core", "stop", "base"]]\n    masks = [(y > 0) if positive_only else np.ones(len(y), dtype=bool) for y in [core_y, stop_y, base_y]]\n    if classification:\n        core_y, stop_y, base_y = [(y > 0).astype(int) for y in [core_y, stop_y, base_y]]\n        if np.unique(core_y).size < 2 or np.unique(base_y).size < 2:\n            return ConstantEstimator(base_y.mean()), 0\n    if masks[0].sum() < 2 or masks[2].sum() < 2:\n        return ConstantEstimator(base_y[masks[2]].mean() if masks[2].any() else 0), 0\n    objective = "binary:logistic" if classification else (\n        "reg:gamma" if positive_only and params["severity_objective"] == "gamma" else "reg:tweedie")\n    power = params.get("severity_power", 1.5) if positive_only else params.get("tweedie_power", 1.5)\n    common = dict(objective=objective, tree_method="hist", max_bin=128, n_jobs=threads, random_state=SEED,\n                  learning_rate=params["learning_rate"], max_depth=params["clf_depth"] if classification else params["reg_depth"],\n                  min_child_weight=params["clf_min_child"] if classification else params["reg_min_child"],\n                  subsample=params["subsample"], colsample_bytree=params["colsample"],\n                  reg_alpha=params["reg_alpha"], reg_lambda=params["reg_lambda"], verbosity=0)\n    if objective == "reg:tweedie":\n        common["tweedie_variance_power"] = power\n    common["eval_metric"] = "logloss" if classification else ("gamma-nloglik" if objective == "reg:gamma" else f"tweedie-nloglik@{power}")\n    estimator = XGBClassifier if classification else XGBRegressor\n    core_w = group_weights(part["core"], params["weight_mode"])[masks[0]]\n    base_w = group_weights(part["base"], params["weight_mode"])[masks[2]]\n    if masks[1].any():\n        early = estimator(n_estimators=max_trees, early_stopping_rounds=30, **common)\n        early.fit(part["core_x"][masks[0]], core_y[masks[0]], sample_weight=core_w,\n                  eval_set=[(part["stop_x"][masks[1]], stop_y[masks[1]])], verbose=False)\n        trees = int(early.best_iteration) + 1\n    else:\n        trees = min(100, max_trees)\n    final = estimator(n_estimators=max(1, trees), **common)\n    final.fit(part["base_x"][masks[2]], base_y[masks[2]], sample_weight=base_w, verbose=False)\n    return final, trees\n\n\nclass RateModel:\n    def __init__(self, pp, features, family):\n        self.pp, self.features, self.family = pp, features, family\n        self.calibrator, self.severity_scale = None, 1.0\n\n    def predict_components(self, frame):\n        if not len(frame):\n            return np.array([]), np.array([]), np.array([])\n        x = self.pp.transform(frame[self.features])\n        if self.family == "tweedie":\n            rate = np.clip(self.regressor.predict(x) * self.severity_scale, 0, 100)\n            return rate, np.full(len(frame), np.nan), np.full(len(frame), np.nan)\n        prob = self.classifier.predict_proba(x)[:, 1]\n        if self.calibrator is not None:\n            prob = self.calibrator.predict_proba(logit(np.clip(prob, 1e-6, 1 - 1e-6)).reshape(-1, 1))[:, 1]\n        severity = np.clip(self.regressor.predict(x) * self.severity_scale, 0, 100)\n        return np.clip(prob * severity, 0, 100), prob, severity\n\n    def predict(self, frame):\n        return self.predict_components(frame)[0]\n\n\ndef fit_rate_model(part, features, params, max_trees=500, threads=4):\n    model = RateModel(part["pp"], features, params["family"])\n    model.diagnostics = {"base_rows": len(part["base"]), "calibration_rows": len(part["cal"])}\n    if params["family"] == "hurdle":\n        model.classifier, trees = _fit_booster(part, params, "occurrence", max_trees, threads)\n        model.diagnostics["classifier_trees"] = trees\n        model.regressor, trees = _fit_booster(part, params, "severity", max_trees, threads)\n        model.diagnostics["severity_trees"] = trees\n    else:\n        model.regressor, trees = _fit_booster(part, params, "combined", max_trees, threads)\n        model.diagnostics["regression_trees"] = trees\n    if params["calibration"] == "calibrated":\n        y = part["cal"][TARGET].to_numpy(float)\n        x = part["cal_x"]\n        if params["family"] == "hurdle":\n            positive = y > 0\n            if positive.sum() >= 20 and (~positive).sum() >= 20:\n                probability = model.classifier.predict_proba(x)[:, 1]\n                model.calibrator = LogisticRegression(C=1.0, max_iter=500).fit(logit(np.clip(probability, 1e-6, 1 - 1e-6)).reshape(-1, 1), positive.astype(int))\n                denominator = np.maximum(model.regressor.predict(x[positive]), 1e-8).sum()\n                model.severity_scale = float(np.clip(y[positive].sum() / denominator, .5, 2))\n                model.diagnostics["calibration"] = "held-out sigmoid occurrence and positive-mean severity scaling"\n            else:\n                model.diagnostics["calibration"] = "skipped: insufficient positive/negative calibration cases"\n        else:\n            denominator = np.maximum(model.regressor.predict(x), 1e-8).sum()\n            model.severity_scale = float(np.clip(y.sum() / denominator, .5, 2))\n            model.diagnostics["calibration"] = "held-out mean scaling"\n    return model\n\n\ndef baseline_oof(dev, splits, alpha=0):\n    pred = np.zeros(len(dev))\n    for train, valid in splits:\n        pred[valid] = HistoricalRate(alpha).fit(dev.iloc[train]).predict(dev.iloc[valid])\n    return pred\n\n\ndef default_params():\n    return dict(family="hurdle", severity_objective="gamma", severity_power=1.5, tweedie_power=1.5,\n                learning_rate=.05, clf_depth=3, reg_depth=3, clf_min_child=5., reg_min_child=1.,\n                subsample=.9, colsample=.9, reg_alpha=.01, reg_lambda=5., weight_mode="profiles", calibration="calibrated")\n\n\ndef cv_predict(dev, partitions, splits, features, params, max_trees=500, threads=4):\n    out, prob = np.zeros(len(dev)), np.full(len(dev), np.nan)\n    for part, (_, valid) in zip(partitions, splits):\n        fitted = fit_rate_model(part, features, params, max_trees, threads)\n        out[valid], prob[valid], _ = fitted.predict_components(dev.iloc[valid])\n    return out, prob\n\n\ndef fingerprint(dev, features, config):\n    digest = hashlib.sha256(pd.util.hash_pandas_object(dev, index=True).to_numpy().tobytes())\n    digest.update(json.dumps([CODE_VERSION, features, config, versions()], sort_keys=True).encode())\n    digest.update(Path(__file__).read_bytes())\n    return digest.hexdigest()\n\n\ndef search_models(dev, features, output_dir, n_trials=12, max_trees=500, n_splits=3, min_profiles=10, threads=4, entity_column=None):\n    import optuna\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    quantity = "production_kg" in dev\n    error_metric = "district_mae_kg" if quantity else "district_mae_pp"\n    config = dict(max_trees=max_trees, n_splits=n_splits, min_profiles=min_profiles, entity_column=entity_column,\n                  ranking_basis="predicted_loss_kg" if quantity else "predicted_loss_pct",\n                  loss_quantity_scope="survey_expanded" if "survey_weight" in dev else "sample_total" if quantity else "unavailable")\n    stamp = fingerprint(dev, features, config)\n    splits = cv_splits(dev, n_splits, entity_column)\n    baseline_predictions, baseline_rows = {}, []\n    for alpha in [0, 50, 200]:\n        name = "Historical exact" if alpha == 0 else f"Historical shrinkage {alpha}"\n        pred = baseline_oof(dev, splits, alpha)\n        baseline_predictions[name] = pred\n        baseline_rows.append({"model": name, "alpha": alpha, **metrics(dev, pred, min_profiles)})\n    zero = metrics(dev, np.zeros(len(dev)), min_profiles)\n    baseline_rows.append({"model": "Always zero (diagnostic only)", "alpha": np.nan, **zero})\n    baselines = pd.DataFrame(baseline_rows)\n    baselines.to_csv(output_dir / "baseline_cv_2024.csv", index=False)\n    if not baselines.ranking_cohorts.max():\n        raise ValueError("No crop-season cohort has enough supported districts and nonconstant outcomes. Ranking selection is not possible.")\n    print("Preparing independent early-stopping/calibration subsets (2024 only)...", flush=True)\n    partitions = [prepare_partition(dev.iloc[tr], dev.iloc[va], features, SEED + i, entity_column) for i, (tr, va) in enumerate(splits)]\n    state_path = output_dir / "search_checkpoint.joblib"\n    study = None\n    if state_path.exists():\n        saved = joblib.load(state_path)  # Load only your own trusted checkpoint.\n        if saved.get("fingerprint") == stamp and saved.get("optuna_version") == optuna.__version__:\n            study = saved["study"]\n            print("Resuming compatible 2024-only search.", flush=True)\n    if study is None:\n        startup = min(4, max(1, n_trials // 3))\n        study = optuna.create_study(directions=["maximize", "minimize"], sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=startup))\n    optuna.logging.set_verbosity(optuna.logging.WARNING)\n    cache_dir = output_dir / ("oof_" + stamp[:12])\n    cache_dir.mkdir(exist_ok=True)\n\n    def objective(trial):\n        started = time.monotonic()\n        params = default_params()\n        params.update(family=trial.suggest_categorical("family", ["hurdle", "tweedie"]),\n                      calibration=trial.suggest_categorical("calibration", ["none", "calibrated"]),\n                      learning_rate=trial.suggest_float("learning_rate", .025, .12, log=True),\n                      clf_depth=trial.suggest_int("clf_depth", 2, 4), reg_depth=trial.suggest_int("reg_depth", 2, 4),\n                      clf_min_child=trial.suggest_float("clf_min_child", 2, 30, log=True),\n                      reg_min_child=trial.suggest_float("reg_min_child", .1, 8, log=True),\n                      subsample=trial.suggest_float("subsample", .75, 1), colsample=trial.suggest_float("colsample", .75, 1),\n                      reg_alpha=trial.suggest_float("reg_alpha", 1e-4, .5, log=True),\n                      reg_lambda=trial.suggest_float("reg_lambda", 1, 30, log=True),\n                      weight_mode=trial.suggest_categorical("weight_mode", ["profiles", "district_balanced"]))\n        if params["family"] == "hurdle":\n            params["severity_objective"] = trial.suggest_categorical("severity_objective", ["gamma", "tweedie"])\n            params["severity_power"] = trial.suggest_float("severity_power", 1.1, 1.8)\n        else:\n            params["tweedie_power"] = trial.suggest_float("tweedie_power", 1.1, 1.8)\n        prediction, probability = cv_predict(dev, partitions, splits, features, params, max_trees, threads)\n        result = metrics(dev, prediction, min_profiles)\n        if not np.isfinite(result["ndcg5_macro"]):\n            raise ValueError("No ranking metric can be evaluated on these folds.")\n        trial.set_user_attr("resolved_params", params)\n        trial.set_user_attr("metrics", {key: value if np.isfinite(value) else None for key, value in result.items()})\n        np.savez_compressed(cache_dir / f"trial_{trial.number}.npz", prediction=prediction, probability=probability)\n        print(f"Trial {trial.number + 1}: NDCG@5={result[\'ndcg5_macro\']:.3f}, {error_metric}={result[error_metric]:.3f}, {time.monotonic() - started:.1f}s", flush=True)\n        return result["ndcg5_macro"], result[error_metric]\n\n    def checkpoint(study, trial):\n        temporary = state_path.with_suffix(".tmp")\n        joblib.dump({"fingerprint": stamp, "study": study, "optuna_version": optuna.__version__}, temporary)\n        os.replace(temporary, state_path)\n        study.trials_dataframe().to_csv(output_dir / "tuning_trials_2024.csv", index=False)\n\n    complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]\n    study.optimize(objective, n_trials=max(0, n_trials - len(complete)), callbacks=[checkpoint], n_jobs=1)\n    # OOF scores here are selection scores, not an unbiased nested-CV estimate.\n    rows = baseline_rows[:-1].copy()\n    choices = {row["model"]: {"kind": "baseline", "alpha": row["alpha"]} for row in rows}\n    predictions = dict(baseline_predictions)\n    for trial in study.trials:\n        if trial.state != optuna.trial.TrialState.COMPLETE:\n            continue\n        path = cache_dir / f"trial_{trial.number}.npz"\n        if path.exists():\n            pred = np.load(path)["prediction"]\n        else:\n            pred, _ = cv_predict(dev, partitions, splits, features, trial.user_attrs["resolved_params"], max_trees, threads)\n        for alpha in [0, 50, 200]:\n            base_name = "Historical exact" if alpha == 0 else f"Historical shrinkage {alpha}"\n            for weight in [.25, .5, .75, 1.]:\n                if weight == 1 and alpha != 0:\n                    continue\n                name = f"Trial {trial.number}: ML={weight:g}, prior alpha={alpha}"\n                blended = weight * pred + (1 - weight) * baseline_predictions[base_name]\n                rows.append({"model": name, **metrics(dev, blended, min_profiles)})\n                predictions[name] = blended\n                choices[name] = {"kind": "blend", "ml_weight": weight, "alpha": alpha, "params": trial.user_attrs["resolved_params"]}\n    comparison = pd.DataFrame(rows)\n    reference = baselines.loc[baselines.model.ne("Always zero (diagnostic only)"), error_metric].min()\n    admissible = comparison.loc[comparison[error_metric] <= reference * 1.10 + 1e-12]\n    winner = admissible.sort_values(["ndcg5_macro", error_metric, "model"], ascending=[False, True, True]).iloc[0]\n    choice = choices[winner.model]\n    lock = {"choice": choice, "features": features, "fingerprint": stamp, "config": config,\n            "selection": winner.to_dict(), "selection_rule": f"2024 OOF: best macro NDCG@5 ranked by {config[\'ranking_basis\']}, with {error_metric} <= 1.10 times the best historical baseline; zero predictor diagnostic only",\n            "versions": versions(), "test_year": 2025, "new_unseen_year_still_required": True}\n    (output_dir / "locked_choice_before_2025.json").write_text(json.dumps(lock, indent=2), encoding="utf-8")\n    comparison.to_csv(output_dir / "candidate_comparison_2024.csv", index=False)\n    return lock, predictions[winner.model], comparison\n\n\nclass BlendedRate:\n    def __init__(self, prior, ml=None, ml_weight=0):\n        self.prior, self.ml, self.ml_weight = prior, ml, ml_weight\n\n    def predict_components(self, frame):\n        rate, prob, severity = self.prior.predict_components(frame)\n        if self.ml is None or self.ml_weight == 0:\n            return rate, prob, severity\n        other, other_prob, _ = self.ml.predict_components(frame)\n        rate = self.ml_weight * other + (1 - self.ml_weight) * rate\n        if np.isfinite(other_prob).all():\n            prob = self.ml_weight * other_prob + (1 - self.ml_weight) * prob\n            severity = np.divide(rate, prob, out=np.zeros(len(frame)), where=prob > 0)\n        else:\n            prob, severity = np.full(len(frame), np.nan), np.full(len(frame), np.nan)\n        return np.clip(rate, 0, 100), prob, severity\n\n    def predict(self, frame):\n        return self.predict_components(frame)[0]\n\n\ndef fit_locked(dev, lock, threads=4):\n    if not dev.year.eq(2024).all():\n        raise ValueError("The evaluation model must fit 2024 only.")\n    choice, config, features = lock["choice"], lock["config"], lock["features"]\n    prior = HistoricalRate(choice["alpha"]).fit(dev)\n    if choice["kind"] == "baseline":\n        return BlendedRate(prior)\n    part = prepare_partition(dev, None, features, SEED + 900, config["entity_column"])\n    ml = fit_rate_model(part, features, choice["params"], config["max_trees"], threads)\n    return BlendedRate(prior, ml, choice["ml_weight"])\n\n\ndef component_metrics(frame, rate, probability, severity):\n    y = frame[TARGET].to_numpy(float)\n    result = {"rows": len(y), "positive_cases": int((y > 0).sum()), "positive_prevalence": float((y > 0).mean())}\n    if np.isfinite(probability).all():\n        result["occurrence_brier"] = float(brier_score_loss(y > 0, probability))\n        result["occurrence_average_precision"] = float(average_precision_score(y > 0, probability)) if (y > 0).any() else np.nan\n    if (y > 0).any() and np.isfinite(severity).all():\n        result["conditional_positive_rmse_pp"] = float(np.sqrt(np.mean((severity[y > 0] - y[y > 0]) ** 2)))\n        result["conditional_mean_predicted_pct"] = float(severity[y > 0].mean())\n        result["conditional_mean_observed_pct"] = float(y[y > 0].mean())\n    return result\n\n\ndef evaluate_locked(dev, test, bundle, lock, output_dir):\n    if not test.year.eq(2025).all():\n        raise ValueError("Temporal evaluation expects 2025 rows only.")\n    output_dir = Path(output_dir)\n    min_profiles = lock["config"]["min_profiles"]\n    ranking_basis = "predicted_loss_kg" if "production_kg" in test else "predicted_loss_pct"\n    scope = "survey_expanded" if "survey_weight" in test else "sample_total" if "production_kg" in test else "unavailable"\n    if ranking_basis != lock["config"].get("ranking_basis", "predicted_loss_pct") or scope != lock["config"].get("loss_quantity_scope", "unavailable"):\n        raise ValueError("Temporal evaluation must use the locked ranking basis and quantity scope; do not mix rates, sample totals and survey-expanded totals.")\n    rate, probability, severity = bundle.predict_components(test)\n    predictors = {"V3 locked candidate": rate,\n                  "Historical exact": HistoricalRate(0).fit(dev).predict(test),\n                  "Historical shrinkage 50": HistoricalRate(50).fit(dev).predict(test),\n                  "Always zero (diagnostic only)": np.zeros(len(test))}\n    rows = []\n    for scope, mask in [("2025 all", np.ones(len(test), dtype=bool)), ("2025 A+B", test.season.isin(["A", "B"]).to_numpy())]:\n        if mask.any():\n            for name, values in predictors.items():\n                rows.append({"scope": scope, "model": name, **metrics(test.loc[mask], values[mask], min_profiles)})\n    comparison = pd.DataFrame(rows)\n    comparison.to_csv(output_dir / "external_validation_2025.csv", index=False)\n    seasons, components = [], []\n    for season in sorted(test.season.unique()):\n        mask = test.season.eq(season).to_numpy()\n        seasons.append({"season": season, "positive_cases": int(test.loc[mask, TARGET].gt(0).sum()), **metrics(test.loc[mask], rate[mask], min_profiles)})\n        components.append({"season": season, **component_metrics(test.loc[mask], rate[mask], probability[mask], severity[mask])})\n    pd.DataFrame(seasons).to_csv(output_dir / "external_validation_by_season_2025.csv", index=False)\n    pd.DataFrame(components).to_csv(output_dir / "component_diagnostics_2025.csv", index=False)\n    by_crop = [{"crop_name": crop, **metrics(part, rate[test.crop_name.eq(crop).to_numpy()], min_profiles)} for crop, part in test.groupby("crop_name")]\n    pd.DataFrame(by_crop).to_csv(output_dir / "external_validation_by_crop_2025.csv", index=False)\n    agg = aggregate_rates(test, rate)\n    risk = agg.rename(columns={"predicted": "predicted_loss_pct", "observed": "observed_loss_pct", "profiles": "n_profiles"})\n    risk["vulnerability_rank"] = risk.groupby(["season", "crop_name"])[ranking_basis].rank(method="min", ascending=False).astype(int)\n    risk["support_level"] = pd.cut(risk.n_profiles, [-1, 9, 29, 99, np.inf], labels=["Limited", "Low", "Moderate", "Strong"]).astype(str)\n    risk.to_csv(output_dir / "harvestguard_2025_v3_vulnerability.csv", index=False)\n    ranking_table(agg, min_profiles).to_csv(output_dir / "ranking_cohorts_2025.csv", index=False)\n    for name, prediction in predictors.items():\n        if name in {"V3 locked candidate", "Historical exact"}:\n            detail = ranking_table(aggregate_rates(test, prediction), min_profiles)\n            detail.to_csv(output_dir / ("ranking_detail_candidate.csv" if name == "V3 locked candidate" else "ranking_detail_baseline.csv"), index=False)\n    profile = test[KEYS + [TARGET]].copy()\n    profile["expected_loss_pct"], profile["occurrence_probability"], profile["conditional_severity_pct"] = rate, probability, severity\n    if "production_kg" in test:\n        quantities = profile_loss_quantities(test, rate)\n        for name in ["production_kg", "survey_weight", "predicted_loss_kg", "observed_loss_kg", "weighted_production_kg", "weighted_predicted_loss_kg", "weighted_observed_loss_kg"]:\n            profile[name] = quantities[name].to_numpy()\n    profile.to_csv(output_dir / "profile_predictions_2025.csv", index=False)\n    calibration = pd.DataFrame({"observed": test[TARGET].to_numpy(), "predicted": rate})\n    calibration["bin"] = pd.qcut(calibration.predicted, 10, duplicates="drop") if np.unique(rate).size > 1 else "constant prediction"\n    cal = calibration.groupby("bin", observed=True).agg(n=("observed", "size"), mean_predicted_pct=("predicted", "mean"), mean_observed_pct=("observed", "mean")).reset_index()\n    cal.to_csv(output_dir / "calibration_2025.csv", index=False)\n    ab = comparison.loc[comparison.scope.eq("2025 A+B")].set_index("model")\n    error_metric = "district_mae_kg" if "production_kg" in test else "district_mae_pp"\n    checks = {"A+B district MAE beats exact history": False, "A+B NDCG beats exact history": False,\n              "A+B top5 overlap beats exact history": False}\n    if len(ab):\n        candidate, baseline = ab.loc["V3 locked candidate"], ab.loc["Historical exact"]\n        checks = {"A+B district MAE beats exact history": bool(candidate[error_metric] < baseline[error_metric]),\n                  "A+B NDCG beats exact history": bool(candidate.ndcg5_macro > baseline.ndcg5_macro),\n                  "A+B top5 overlap beats exact history": bool(candidate.top5_overlap_macro > baseline.top5_overlap_macro)}\n    pd.DataFrame([{"check": k, "passed": v} for k, v in checks.items()]).to_csv(output_dir / "readiness_checks.csv", index=False)\n    return comparison, risk, cal, checks\n\n\ndef drift_report(dev, test, features):\n    rows = []\n    for name in features:\n        if name in CATS:\n            rows.append({"feature": name, "kind": "category", "unseen_2025_fraction": float((~test[name].isin(dev[name].unique())).mean())})\n        else:\n            std = dev[name].std()\n            rows.append({"feature": name, "kind": "numeric", "mean_2024": dev[name].mean(), "mean_2025": test[name].mean(),\n                         "standardized_mean_shift": (test[name].mean() - dev[name].mean()) / std if pd.notna(std) and std > 0 else np.nan,\n                         "missing_2024_fraction": dev[name].isna().mean(), "missing_2025_fraction": test[name].isna().mean()})\n    return pd.DataFrame(rows)\n'
module_path = WORK_DIR / 'harvestguard_model_v3.py'
module_path.write_text(MODULE_SOURCE, encoding='utf-8')
sys.path.insert(0, str(WORK_DIR.resolve()))
import importlib
import harvestguard_model_v3 as hg
hg = importlib.reload(hg)
import numpy as np, pandas as pd, joblib, json, shutil, zipfile
import matplotlib.pyplot as plt
print(hg.versions())

## Data and target audit

The existing target is accepted only if finite and within 0–100%. Missing outcomes must never be silently converted to zero.
The cleaned CSV cannot establish whether stage quantities overlap, whether units agree, or whether reported zero means no loss,
not applicable, not asked or missing. Check those definitions against the source questionnaires and cleaning code.

If original quantities are available, configure their columns above to check the arithmetic.
The four quantities must refer to the **same crop, profile, period and unit**. A successful arithmetic check does not establish
that components are mutually exclusive or that a loss was measured correctly.

Quantity aggregation is required in this workflow: `loss_kg_i = predicted_pct_i / 100 * production_kg_i`. Sum profile losses for each season/crop/district. When expansion weights are configured, sum `weight_i * loss_kg_i` and `weight_i * production_kg_i`. The district rate is `100 * total_loss_kg / total_production_kg`; never multiply an unweighted mean rate by total production. Missing or zero production stops the run rather than producing a partial ranking.

Illustration: 30% of 1,000 kg is 300 kg lost; 4% of 100,000 kg is 4,000 kg lost. The second district ranks first by quantity. These are example numbers, not Rwanda observations.

Crop eligibility is computed from 2024 only. Very few positive cases are explicitly flagged;
for example, V2 eligibility allowed Pea with 11 positives and Groundnut with 16. Such evidence cannot support an elaborate crop-specific model.

In [ ]:
# @title 4. Load and audit development data
if not DATA_PATH.exists():
    raise FileNotFoundError(f'CSV not found: {DATA_PATH}. Edit DATA_PATH in the configuration cell.')
raw = pd.read_csv(DATA_PATH)
print('Available CSV columns:', raw.columns.tolist())
print('Configured production column:', repr(PRODUCTION_COLUMN))
if SOURCE_QUANTITIES is not None:
    arithmetic = hg.audit_target_quantities(raw, SOURCE_QUANTITIES)
    arithmetic.to_csv(OUTPUT_DIR / 'target_arithmetic_audit.csv', index=False)
    if arithmetic.mismatch.any():
        raise ValueError('Saved target disagrees with the configured source quantities. Resolve units/target construction before fitting.')
else:
    print('Optional target-arithmetic audit skipped (SOURCE_QUANTITIES is unset). This is informational and does not prevent scoring; production mapping is checked separately below.')

raw = hg.prepare_production(raw, PRODUCTION_COLUMN, PRODUCTION_UNIT, SURVEY_WEIGHT_COLUMN)
data = hg.prepare_data(raw, FEATURE_MODE)
print('Quantity scope:', 'survey_expanded' if SURVEY_WEIGHT_COLUMN else 'sample_total')
features = hg.feature_columns(FEATURE_MODE)
dev_all = data.loc[data.year.eq(2024)].copy()
eligibility = hg.crop_evidence(dev_all)
eligibility.to_csv(OUTPUT_DIR / 'crop_eligibility_2024.csv', index=False)
eligible_crops = eligibility.loc[eligibility.eligible, 'crop_name'].tolist()
if not eligible_crops:
    raise ValueError('No crop meets the development evidence rule. Fix the dataset or use a simpler descriptive analysis.')
dev = dev_all.loc[dev_all.crop_name.isin(eligible_crops)].reset_index(drop=True)
display(eligibility.sort_values('positives', ascending=False))
print('2024 eligible profiles:', len(dev))
print('2024 positive cases:', int(dev[hg.TARGET].gt(0).sum()))
print('2024 zero share:', round(dev[hg.TARGET].eq(0).mean() * 100, 2), '%')
print('Numeric missingness is imputed inside training partitions only:')
display(dev[features].isna().mean().sort_values(ascending=False).head(12))
print('Duplicate full feature rows:', int(dev.duplicated(features, keep=False).sum()))
print('Identical covariates can describe different farmers; do not delete them without verified entity identifiers.')
print('No 2025 outcomes have been used for tuning, calibration or crop eligibility.')

## Search and lock using 2024 only

The target is an **expected percentage**, not a binary accuracy score.
Hurdle expected loss equals occurrence probability × conditional positive severity.
For positive severity, Gamma/Tweedie models use the original percentage-point units.

Inside each fold: grouped calibration subset → separate base/core/early-stop subsets → refit base model with chosen tree count
→ optional calibration on the reserved calibration subset → predict outer validation rows.
Outer validation labels do not choose the number of trees or the calibrator.
No oversampling or positive-class weighting is used to inflate occurrence probabilities.
Optional district-balanced training weights serve the ranking task; they are not survey weights.

Selection maximizes macro district NDCG@5 using observed and predicted **kg lost**, subject to district MAE in kg within 10% of the best historical CV baseline. Profile rate MAE and production-weighted district rate MAE remain diagnostic metrics. Candidate and baseline quantities use the same production, weights and rows.
That tolerance is a declared development rule, not a proven optimal trade-off.
2024 OOF scores reused for tuning/selection are **selection estimates**, not an unbiased nested-CV performance claim.

The grouped CV historical baseline cannot retrieve an exact district/crop/season mean if that entire group is held out.
In 2025, exact 2024 group history is often available. This is why the stronger temporal baseline must still be beaten.

In [ ]:
# @title 5. Tune, compare baselines/blends, and lock the candidate
lock, oof_prediction, comparison_2024 = hg.search_models(
    dev, features, OUTPUT_DIR, n_trials=TRIALS, max_trees=MAX_TREES,
    n_splits=N_SPLITS, min_profiles=MIN_PROFILES_PER_DISTRICT,
    threads=THREADS, entity_column=ENTITY_COLUMN,
)
display(comparison_2024.sort_values(['ndcg5_macro', 'district_mae_kg'], ascending=[False, True]).head(15))
print('LOCKED:', lock['selection']['model'])
print('The lock is saved before temporal outcomes are evaluated.')

## Fit, save, reload and predict

The base ML model uses an internal 2024 training subset; reserved 2024 rows supply calibration.
The historical component uses 2024 data only. A blend may win, or the selected model may be a historical baseline.
Do not label a baseline winner as Hurdle XGBoost.

The reload check confirms that the exported artifact actually produces the same numerical predictions.
It checks implementation consistency, not predictive accuracy.
Load only your own trusted joblib files. The included Python module must be importable when loading the bundle.

In [ ]:
# @title 6. Fit the locked model and verify saved inference
bundle = hg.fit_locked(dev, lock, threads=THREADS)
MODEL_FILE = OUTPUT_DIR / 'model_v3_candidate.joblib'
joblib.dump(bundle, MODEL_FILE)
RELOADED = joblib.load(MODEL_FILE)
probe = dev.head(min(100, len(dev))).copy()
np.testing.assert_allclose(bundle.predict(probe), RELOADED.predict(probe), rtol=1e-7, atol=1e-8)
print('Saved model reload check: passed.')
print('ML component:', bundle.ml.family if bundle.ml is not None else 'none: historical baseline selected')
if bundle.ml is not None:
    print('Internal fit/calibration details:', bundle.ml.diagnostics)

## 2025 temporal validation — evaluate, do not retune

Ranking metrics now evaluate summed loss quantities in kg, not average percentages. Legacy rate-based ranking scores cannot be compared directly with these quantity-based scores.

All candidates/baselines below are applied to the same 2025 rows. Unsupported/constant cohorts are reported rather than assigned a perfect ranking score.
Top-five overlap handles cutoff ties fractionally, so its values may differ from V2's arbitrary tie ordering.
The default ranking analysis requires 10 profiles per district group. **Do not directly compare V3 numbers with V2 numbers
that use a different cohort/support rule.** Compare the candidate and baseline side by side within this new report.

MAE alone rewards predicting zero on a zero-heavy target. The always-zero predictor is therefore a diagnostic,
not an eligible expected-loss champion. Use expected-rate calibration, RMSE, district ranking and subgroup results together.
Occurrence average precision (AP) must be compared with observed positive prevalence; classification accuracy is uninformative here.

Season C had zero reported positive losses in the supplied 2025 export. Audit its source/cleaning/skip rules.
Do not hard-code zero risk from that validation outcome.

In [ ]:
# @title 7. Temporal comparison, ranking, calibration and component diagnostics
test_all = data.loc[data.year.eq(2025)].copy()
test = test_all.loc[test_all.crop_name.isin(eligible_crops)].reset_index(drop=True)
if not len(test):
    raise ValueError('No eligible crops appear in 2025.')
print('2025 excluded crop profiles:', len(test_all) - len(test))
external, vulnerability, calibration, checks = hg.evaluate_locked(dev, test, RELOADED, lock, OUTPUT_DIR)
display(external)
display(pd.read_csv(OUTPUT_DIR / 'external_validation_by_season_2025.csv'))
display(pd.read_csv(OUTPUT_DIR / 'component_diagnostics_2025.csv'))
display(pd.read_csv(OUTPUT_DIR / 'external_validation_by_crop_2025.csv'))
display(pd.DataFrame([{'check': key, 'passed': value} for key, value in checks.items()]))
if all(checks.values()):
    print('Candidate beats the exact historical baseline on these three temporal point metrics. This is not proof of a stable improvement or intervention impact.')
else:
    print('Candidate does not pass every temporal comparison. Keep the stronger baseline/model; do not claim improvement.')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(calibration.mean_predicted_pct, calibration.mean_observed_pct, marker='o')
limit = max(calibration.mean_predicted_pct.max(), calibration.mean_observed_pct.max(), .01)
axes[0].plot([0, limit], [0, limit], '--', color='gray')
axes[0].set(xlabel='Mean predicted loss (%)', ylabel='Mean observed loss (%)', title='2025 expected-rate calibration')
view = external.loc[external.scope.eq('2025 A+B')]
axes[1].barh(view.model, view.ndcg5_macro)
axes[1].set(xlabel='Macro NDCG@5', title='Same-row 2025 A+B ranking by kg lost')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'temporal_validation.png', dpi=160, bbox_inches='tight')
plt.show()

## Drift and evidence limits

Feature shifts and unseen categories are diagnostic leads, not proof of the cause of a model error.
Review stage definitions, source units, district/weather joins, missing-value codes and changing survey questions.
Repeated households and survey sampling clusters require identifiers/design information to assess independence correctly.

One development year cannot teach stable same-season year-to-year changes reliably.
For the next data upgrade, add more independent years, compatible production quantities, verified survey weights,
and pre-outcome measurements of storage method, moisture, drying and handling where available.
This run evaluates loss quantity priorities. Unweighted quantities cover the sample; verified survey expansion weights are needed to estimate population burden from survey profiles. Eligibility filtering also limits coverage to supported crops. Adding production does not establish model accuracy or intervention effectiveness.
Storage measurements and model attributions are associations, not evidence of intervention effects.

No formal uncertainty interval is invented from unknown survey design. Read point-metric gains cautiously;
seek design-aware uncertainty and a new year before investment or national allocation claims.

In [ ]:
# @title 8. Drift report and export manifest
drift = hg.drift_report(dev, test, features)
display(drift)
drift.to_csv(OUTPUT_DIR / 'feature_drift_2024_to_2025.csv', index=False)
season_audit = data.groupby(['year', 'season']).agg(
    rows=(hg.TARGET, 'size'), positive_cases=(hg.TARGET, lambda y: int((y > 0).sum())),
    mean_loss_pct=(hg.TARGET, 'mean')).reset_index()
season_audit.to_csv(OUTPUT_DIR / 'target_by_year_season.csv', index=False)
display(season_audit)

manifest = {
    'schema_version': 1, 'candidate_version': hg.CODE_VERSION,
    'champion': lock['selection']['model'], 'target': hg.TARGET, 'target_unit': 'percent of production, 0 to 100',
    'features': features, 'feature_policy': FEATURE_MODE, 'eligible_crops': eligible_crops,
    'model_file': MODEL_FILE.name, 'module_file': 'harvestguard_model_v3.py',
    'development_year': 2024, 'temporal_validation_year': 2025,
    '2025_previously_inspected': True, 'new_uninspected_period_required': True,
    'cv_grouping': ENTITY_COLUMN or 'district x crop x season',
    'ranking_min_profiles': MIN_PROFILES_PER_DISTRICT,
    'temporal_checks': checks, 'survey_weights_available': SURVEY_WEIGHT_COLUMN is not None,
    'production_burden_available': True, 'ranking_basis': 'predicted_loss_kg',
    'loss_quantity_scope': 'survey_expanded' if SURVEY_WEIGHT_COLUMN else 'sample_total',
    'production_source_column': PRODUCTION_COLUMN, 'production_source_unit': PRODUCTION_UNIT,
    'survey_weight_source_column': SURVEY_WEIGHT_COLUMN,
    'aggregation': 'Sum profile predicted_pct / 100 * production_kg, applying survey expansion weights to both production and loss when configured. District percentage = 100 * summed loss kg / summed production kg.',
    'model_ready_for_national_allocation': False, 'versions': hg.versions(),
    'uncertainty': 'point estimates only; survey-design uncertainty not established',
    'ai_contract': 'Use exported aggregate predictions and actual evidence. Do not invent local drivers or treatment benefits.'
}
(OUTPUT_DIR / 'model_v3_metadata.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
source_output = OUTPUT_DIR / 'harvestguard_model_v3.py'
if source_output.resolve() != module_path.resolve():
    shutil.copy2(module_path, source_output)
import optuna
pins = hg.versions()
requirements = '\n'.join([f"numpy=={pins['numpy']}", f"pandas=={pins['pandas']}",
    f"scipy=={pins['scipy']}", f"scikit-learn=={pins['sklearn']}",
    f"xgboost=={pins['xgboost']}", f"optuna=={optuna.__version__}", f"joblib=={joblib.__version__}"]) + '\n'
(OUTPUT_DIR / 'requirements_model_v3.txt').write_text(requirements, encoding='utf-8')
print('All outputs saved:', OUTPUT_DIR)

## Score genuinely new feature rows later

This is real saved-model inference: it does not read the existing dashboard CSV and does not need new loss labels.
Supply the same feature definitions, units and prediction-time policy used for training.
For unknown/unsupported crops, obtain evidence and retrain; do not silently extrapolate a trusted recommendation.
Keep compatible production on incoming rows to convert the model rates to quantities. The dashboard supports precomputed V3 quantity exports. Updating its bundle is separate from fitting and validating the candidate.

In [ ]:
# @title 9. Verify unlabeled inference (then adapt this example to your new CSV)
unlabeled_example = test.head(min(30, len(test))).drop(columns=[hg.TARGET, 'year'])
incoming = hg.prepare_features(unlabeled_example, manifest['features'])
if not incoming.crop_name.isin(manifest['eligible_crops']).all():
    raise ValueError('Incoming data includes a crop outside the validated eligibility set.')
expected_rate, occurrence, conditional_severity = RELOADED.predict_components(incoming)
np.testing.assert_allclose(expected_rate, RELOADED.predict(test.head(len(incoming))))
inference_example = incoming[hg.KEYS].copy()
inference_example['expected_loss_pct'] = expected_rate
inference_example['occurrence_probability'] = occurrence
inference_example['conditional_severity_pct'] = conditional_severity
inference_example['production_kg'] = incoming.production_kg.to_numpy()
inference_example['predicted_loss_kg'] = expected_rate / 100 * incoming.production_kg.to_numpy()
display(inference_example.head(10))
quantity_priorities = hg.aggregate_losses(incoming, expected_rate)
display(quantity_priorities.sort_values(['season', 'crop_name', 'priority_rank', 'district']))
print('This small inference example covers only the incoming rows, not complete districts.')
print('If a single-stage Tweedie model won, occurrence/severity are unavailable (NaN), not fabricated.')
print('For a hurdle/prior mixture, conditional severity is the coherent mixture rate divided by mixture probability.')

## Download the candidate package

The ZIP includes the actual model, its source, locked configuration, environment versions and summary reports.
It excludes checkpoints/OOF caches and individual-profile prediction rows; those remain private in your output directory.
The district vulnerability CSV retains its V3 filename and adds production_kg, predicted/observed_loss_kg, predicted/observed_loss_tonnes, priority_rank and loss_quantity_scope. Percentages are now production-weighted; vulnerability_rank follows kg priorities for compatibility. The current app must not automatically
claim its champion is Hurdle XGBoost if this notebook selected a baseline, single-stage model or blend.

After running, share the summary tables (especially `external_validation_2025.csv`, `component_diagnostics_2025.csv`,
`target_by_year_season.csv` and `model_v3_metadata.json`) to decide the next step. No API key or raw farmer records are needed.

In [ ]:
# @title 10. Build the ZIP and download in Colab
ZIP_PATH = OUTPUT_DIR.parent / (OUTPUT_DIR.name + '.zip')
excluded = {'profile_predictions_2025.csv', 'search_checkpoint.joblib', 'target_arithmetic_audit.csv'}
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file() and path.name not in excluded and path.suffix in {'.csv', '.json', '.py', '.joblib', '.txt', '.png'}:
            archive.write(path, arcname=OUTPUT_DIR.name + '/' + path.name)
print('Candidate package:', ZIP_PATH)
if IN_COLAB and AUTO_DOWNLOAD:
    from google.colab import files
    files.download(str(ZIP_PATH))

## Method references

- [Optuna TPE startup behavior](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.TPESampler.html): startup trials use random sampling, so the startup budget must be smaller than the total search budget.
- [XGBoost parameters](https://xgboost.readthedocs.io/en/stable/parameter.html): Gamma/Tweedie objectives and Hessian-based minimum child weight. Classifier and severity stages need not share that constraint.
- [scikit-learn calibration](https://scikit-learn.org/stable/modules/calibration.html): calibrate using predictions independent of the calibration labels' model fit.
- [scikit-learn R²](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html): negative R² is possible; zero is the constant evaluation-mean reference for nonconstant outcomes, not a deployable oracle.

No synthetic smoke-test metric should be reported as real Rwanda model performance.